# 00. Adathalmaz Áttekintés és Minőségi Riport

**TDK Kutatási Téma**: Budapest Főváros X. kerület (Kőbánya) lakóingatlan-piacának komplex térökonometriai, hedonikus és gépi tanulásos vizsgálata.

**Adatbázis forrása**: Az ingatlan.com kínálati adatbázisából kinyert és térinformatikailag dúsított mestertábla (`kobanya_ingatlan_szamitott_master.parquet`).
Az adatbázis 1 320 darab egyedi hirdetést és 169 strukturált változót tartalmaz, beleértve a hálózati elérhetőségi mutatókat, GIS távolságokat és épületfizikai paramétereket.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

# Adathalmaz betöltése
df = load_szamitott_master()
print(f"Sikeresen betöltve: {len(df)} sor, {len(df.columns)} oszlop.")

### 1. Alapvető Adatbázis KPI Mutatók
Az adathalmaz globális mérete, az eladó és kiadó szegmensek aránya, valamint a garantált térbeli pontosság.

In [ ]:
n_total = len(df)
n_elado = len(df[df['listing_type'] == 'elado'])
n_kiado = len(df[df['listing_type'] == 'kiado'])
n_pontos = len(df[df['minta_garantalt_pontos'] == 1])
median_nm_ar = df[df['listing_type'] == 'elado']['nm_ar_huf'].median()
atlag_alapterulet = df['alapterulet_nm'].mean()

kpi_cards = [
    ("Összes Hirdetés", f"{n_total:,} db".replace(',', ' '), "Teljes adatbázis", "#1e3a8a"),
    ("Eladó Lakások", f"{n_elado:,} db".replace(',', ' '), f"{n_elado/n_total*100:.1f}% arány", "#2563eb"),
    ("Kiadó Lakások", f"{n_kiado:,} db".replace(',', ' '), f"{n_kiado/n_total*100:.1f}% arány", "#d97706"),
    ("Garantált Pontos GIS", f"{n_pontos:,} db".replace(',', ' '), f"{n_pontos/n_total*100:.1f}% koordinátás", "#16a34a"),
    ("Eladó Medián Ár/m²", fmt_huf(median_nm_ar), "Kínálati fajlagos ár", "#9333ea"),
    ("Átlagos Alapterület", f"{atlag_alapterulet:.1f} m²", "Kínálati átlag", "#0891b2"),
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Térbeli és Hirdetéstípus Megoszlás Városrészenként
A kőbányai városrészek közötti kínálati volumen megoszlása eladó és kiadó bontásban.

In [ ]:
df_counts = df.groupby(['varosresz', 'listing_type']).size().reset_index(name='count')
df_counts['listing_nev'] = df_counts['listing_type'].map({'elado': 'Eladó', 'kiado': 'Kiadó'})

fig1 = px.bar(
    df_counts,
    x='varosresz',
    y='count',
    color='listing_nev',
    barmode='group',
    title='Hirdetések megoszlása városrészenként és típusonként',
    labels={'varosresz': 'Városrész', 'count': 'Hirdetések száma', 'listing_nev': 'Típus'},
    color_discrete_map={'Eladó': '#2563eb', 'Kiadó': '#f59e0b'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-30, height=450)
fig1.show()

### 3. Változók Kitöltöttsége és Adatminőség
A modellezés szempontjából kritikus változók hiányzó adatainak aránya.

In [ ]:
kulcs_valtozok = [
    'price_huf', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 
    'emelet_szam', 'allapot', 'epites_eve_kategoria', 'is_panel', 
    'geokodolt_lat', 'tavolsag_mazsa_halozati_m', 'tavolsag_metro_halozati_m'
]
valtozo_nevek = {
    'price_huf': 'Ár (HUF)',
    'nm_ar_huf': 'Négyzetméterár',
    'alapterulet_nm': 'Alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám',
    'emelet_szam': 'Emelet',
    'allapot': 'Műszaki állapot',
    'epites_eve_kategoria': 'Építés éve kat.',
    'is_panel': 'Panelszerkezet',
    'geokodolt_lat': 'Geokódolt koordináta',
    'tavolsag_mazsa_halozati_m': 'Mázsa tér távolság',
    'tavolsag_metro_halozati_m': 'Metró távolság'
}

kitoltottseg = []
for col in kulcs_valtozok:
    if col in df.columns:
        pct = (df[col].notna().sum() / len(df)) * 100
        kitoltottseg.append({'Valtozo': valtozo_nevek.get(col, col), 'Kitoltottseg_pct': pct})

df_qual = pd.DataFrame(kitoltottseg).sort_values('Kitoltottseg_pct', ascending=True)

fig2 = px.bar(
    df_qual,
    x='Kitoltottseg_pct',
    y='Valtozo',
    orientation='h',
    title='Kulcsváltozók kitöltöttségi aránya (%)',
    labels={'Kitoltottseg_pct': 'Kitöltöttség (%)', 'Valtozo': 'Változó'},
    color='Kitoltottseg_pct',
    color_continuous_scale='Blues',
    range_x=[0, 105],
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 4. Interaktív Adathalmaz Szűrő és Ellenőrző Pult
A vezérlőkkel dinamikusan tesztelhetők az alminták (eladó, kiadó, pontos geokódolt, városrész szerinti szűrés).

In [ ]:
# Interaktív vezérlők definiálása
w_tipus = widgets.RadioButtons(options=['Mindkettő', 'eladó', 'kiadó'], value='Mindkettő', description='Hirdetés:')
w_pontos = widgets.Checkbox(value=False, description='Csak pontos koordinátás (N=296)')
w_varosreszek = widgets.SelectMultiple(
    options=VAROSRESZEK,
    value=tuple(VAROSRESZEK[:3]),
    description='Városrészek:'
)

out_summary = widgets.Output()

def frissit_nezopont(*args):
    dff = df.copy()
    if w_tipus.value != 'Mindkettő':
        dff = dff[dff['listing_type'] == w_tipus.value]
    if w_pontos.value:
        dff = dff[dff['minta_garantalt_pontos'] == 1]
    if w_varosreszek.value:
        dff = dff[dff['varosresz'].isin(w_varosreszek.value)]
        
    with out_summary:
        clear_output(wait=True)
        sub_kpis = [
            ("Szűrt Elemek", f"{len(dff):,} db".replace(',', ' '), "Aktuális minta", "#1e3a8a"),
            ("Szűrt Átlagár", fmt_mft(dff['price_huf'].mean() / 1e6 if dff['price_huf'].notna().any() else 0), "Átlagos kínálat", "#059669"),
            ("Szűrt Ár/m²", fmt_huf(dff['nm_ar_huf'].median() if dff['nm_ar_huf'].notna().any() else 0), "Medián fajlagos ár", "#7c3aed"),
            ("Városrészek", f"{len(w_varosreszek.value)} db", "Kiválasztva", "#d97706")
        ]
        display(HTML(kpi_grid_html(sub_kpis)))
        if len(dff) > 0:
            preview_cols = [c for c in ['listing_id', 'cim_teljes', 'ar_millio_ft', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'varosresz'] if c in dff.columns]
            display(HTML(dff[preview_cols].head(5).to_html(classes='table table-striped', index=False)))

w_tipus.observe(frissit_nezopont, names='value')
w_pontos.observe(frissit_nezopont, names='value')
w_varosreszek.observe(frissit_nezopont, names='value')

display(widgets.HBox([w_tipus, w_pontos, w_varosreszek]))
display(out_summary)
frissit_nezopont()